# Тема 1. Анализ данных с Pandas

## Дополнительные материалы (самостоятельное чтение)

На занятии не разбираем — но эти темы регулярно всплывают в реальных задачах: данные редко приходят одним файлом, и в них почти всегда есть даты.

1. NumPy — на чём построен Pandas
2. Объединение таблиц: `concat`, `merge`
3. Работа с датами: `pd.to_datetime`, `.dt`-аксессор

---
## 1. NumPy — на чём построен Pandas

NumPy — библиотека для быстрых вычислений с массивами чисел. Каждая числовая колонка `DataFrame` внутри хранится как NumPy-массив, поэтому операции над колонками такие быстрые.

**Главная идея:** операции применяются сразу ко всему массиву (*векторизация*), без явного цикла `for`.

In [1]:
import numpy as np
import pandas as pd
import time

n = 1_000_000
python_list = list(range(n))
numpy_array = np.arange(n)

t0 = time.time()
doubled_list = [x * 2 for x in python_list]
t_list = time.time() - t0

t0 = time.time()
doubled_array = numpy_array * 2
t_array = time.time() - t0

print(f"Python list: {t_list * 1000:.1f} мс")
print(f"NumPy array: {t_array * 1000:.1f} мс")
print(f"Ускорение: {t_list / t_array:.0f}x")

Python list: 29.6 мс
NumPy array: 1.7 мс
Ускорение: 17x


Другие вещи, которые полезно знать про NumPy (см. [официальную документацию](https://numpy.org/doc/stable/)):

- **Срезы** `a[2:9:3]`, индексация с конца `a[-1]`, разворот `a[::-1]` — работает как со списками
- **Булева маска**: `a[a % 3 == 0]` — фильтрация массива по условию, без цикла
- **Многомерные массивы**: `a.shape`, `a.reshape(...)`, индексация `a[i, j]`
- **Broadcasting**: операции между массивами разной формы (например, массив + число) без ручного дублирования данных
- **Агрегаты по осям**: `.sum(axis=0)` — по столбцам, `.sum(axis=1)` — по строкам, в 2D-массиве или матрице

---
## 2. Объединение таблиц: `concat` и `merge`

На практике данные часто приходят из разных источников — например, отдельно список клиентов и отдельно список их заказов. Их нужно сначала объединить и только потом анализировать вместе.

- **`pd.concat([df1, df2])`** — «приклеить» одну таблицу под/рядом с другой (одинаковые колонки → строки друг под другом)
- **`pd.merge(df1, df2, on='key')`** — SQL-style join по ключевой колонке (собрать строки в одну по общему полю, например `customer_id`)

In [2]:
customers = pd.DataFrame(
    {
        "customer_id": [1, 2, 3, 4],
        "name": ["Аня", "Борис", "Вика", "Гриша"],
        "city": ["Москва", "Казань", "Москва", "Уфа"],
    }
)
customers

,customer_id,name,city
0,1,Аня,Москва
1,2,Борис,Казань
2,3,Вика,Москва
3,4,Гриша,Уфа


**`concat`** — если появились ещё клиенты в такой же табличной структуре, просто приклеиваем их снизу:

In [3]:
new_customers = pd.DataFrame(
    {"customer_id": [5, 6], "name": ["Дима", "Женя"], "city": ["Казань", "Москва"]}
)
pd.concat([customers, new_customers], ignore_index=True)

,customer_id,name,city
0,1,Аня,Москва
1,2,Борис,Казань
2,3,Вика,Москва
3,4,Гриша,Уфа
4,5,Дима,Казань
5,6,Женя,Москва


**`merge`** — а вот заказы лежат в отдельной таблице, и у клиента №3 заказов нет, а заказ №105 сделан клиентом, которого нет в списке `customers` (например, его уже удалили):

In [4]:
orders = pd.DataFrame(
    {
        "order_id": [101, 102, 103, 104, 105],
        "customer_id": [1, 1, 2, 4, 7],
        "amount": [1200, 800, 450, 2200, 300],
    }
)
orders

,order_id,customer_id,amount
0,101,1,1200
1,102,1,800
2,103,2,450
3,104,4,2200
4,105,7,300


Тип `merge` определяет, что делать с несовпадающими строками:

- `inner` — только те `customer_id`, что есть в обеих таблицах
- `left` — все строки из левой таблицы + совпадения из правой (несовпавшие поля — `NaN`)
- `right` — симметрично, все из правой
- `outer` — вообще все строки из обеих таблиц

In [5]:
pd.merge(customers, orders, on="customer_id", how="inner")

,customer_id,name,city,order_id,amount
0,1,Аня,Москва,101,1200
1,1,Аня,Москва,102,800
2,2,Борис,Казань,103,450
3,4,Гриша,Уфа,104,2200


In [6]:
pd.merge(customers, orders, on="customer_id", how="left")

,customer_id,name,city,order_id,amount
0,1,Аня,Москва,101.0,1200.0
1,1,Аня,Москва,102.0,800.0
2,2,Борис,Казань,103.0,450.0
3,3,Вика,Москва,NaN,NaN
4,4,Гриша,Уфа,104.0,2200.0


В `left`-join Вика (id=3, заказов нет) осталась в таблице с `NaN` в колонках заказа, а заказ id=7 (без клиента) пропал — потому что мы взяли за основу таблицу клиентов. Попробуйте самостоятельно `how='outer'` и посмотрите, что получится.

---
## 3. Работа с датами

Если колонка с датой хранится как текст, её стоит явно перевести в тип `datetime64` — тогда становятся доступны удобные операции сравнения, сортировки и аксессор `.dt`.

In [7]:
sales = pd.DataFrame(
    {
        "date": pd.date_range("2025-01-01", periods=14, freq="D"),
        "amount": [12, 15, 9, 20, 25, 30, 28, 14, 11, 18, 22, 26, 31, 19],
    }
)
sales["date"] = pd.to_datetime(sales["date"])
sales.head()

,date,amount
0,2025-01-01,12
1,2025-01-02,15
2,2025-01-03,9
3,2025-01-04,20
4,2025-01-05,25


**`.dt`-аксессор** достаёт компоненты даты:

In [8]:
sales["day_of_week"] = sales["date"].dt.day_name()
sales["is_weekend"] = sales["date"].dt.dayofweek >= 5
sales.head(7)

,date,amount,day_of_week,is_weekend
0,2025-01-01,12,Wednesday,False
1,2025-01-02,15,Thursday,False
2,2025-01-03,9,Friday,False
3,2025-01-04,20,Saturday,True
4,2025-01-05,25,Sunday,True
5,2025-01-06,30,Monday,False
6,2025-01-07,28,Tuesday,False


Фильтрация по дате работает через обычное сравнение (даты сравниваются как числа):

In [9]:
sales[sales["date"] >= "2025-01-10"]

,date,amount,day_of_week,is_weekend
9,2025-01-10,18,Friday,False
10,2025-01-11,22,Saturday,True
11,2025-01-12,26,Sunday,True
12,2025-01-13,31,Monday,False
13,2025-01-14,19,Tuesday,False


In [10]:
sales[sales["is_weekend"]]

,date,amount,day_of_week,is_weekend
3,2025-01-04,20,Saturday,True
4,2025-01-05,25,Sunday,True
10,2025-01-11,22,Saturday,True
11,2025-01-12,26,Sunday,True


---
## Итог

Это были дополнительные инструменты — возвращайтесь к ним по мере необходимости в реальных задачах. Ядро того, что нужно знать для практики и дальнейших тем курса — в `lesson01_pandas_theory.ipynb`.